In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))
from diffusion_generators import train_tabddpm, train_forestdiffusion

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
breast_cancer = fetch_ucirepo(id=17)

X = breast_cancer.data.features
y = breast_cancer.data.targets

cancer_data = pd.concat([X, y], axis=1)

target_col = cancer_data.columns[-1]

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(cancer_data)

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []


In [3]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ---------------------------------------------------
# TRAIN / TEST SPLIT (NO LEAKAGE)
# ---------------------------------------------------

train_real, test_real = train_test_split(
    cancer_data,
    test_size=TEST_SIZE,
    stratify=cancer_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)


================ SINGLE RUN ================
Training TabDDPM...
[0]
32
{'num_classes': 2, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(32)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.3055 Sum: 0.3055
Step 1000/1000 MLoss: 0.0 GLoss: 0.2773 Sum: 0.2773
mlp
Sample timestep    0
Discrete cols: []
Num shape:  (1000, 30)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 152.02it/s]|
Column Shapes Score: 36.88%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:03<00:00, 153.32it/s]|
Column Pair Trends Score: 80.19%

Overall Score (Average): 58.53%

TabDDPM: 0.5853


In [4]:
# ForestDiffusion

try:
    import traceback

    print("Training ForestDiffusion...")
    synthetic_forestdiffusion = train_forestdiffusion(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_forestdiffusion,
        metadata=train_metadata,
    )

    scores["ForestDiffusion"] = quality.get_score()

    print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))

    del synthetic_forestdiffusion

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("ForestDiffusion Failed:")
    traceback.print_exc()


Training ForestDiffusion...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 31/31 [00:00<00:00, 948.84it/s]|
Column Shapes Score: 94.66%

(2/2) Evaluating Column Pair Trends: |██████████| 465/465 [00:01<00:00, 424.08it/s]|
Column Pair Trends Score: 96.6%

Overall Score (Average): 95.63%

ForestDiffusion: 0.9563


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': LinearSVC(max_iter=2000, dual='auto', random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=[42,43,44,45,46,47,48,49,50,51]
):

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=y_train
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=y_test
            )

            scaler = StandardScaler().fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test,
                    y_pred,
                    pos_label="M",
                    average="binary",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test,
                    y_pred,
                    pos_label="M",
                    average="binary",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test,
                    y_pred,
                    pos_label="M",
                    average="binary",
                    zero_division=0
                )
            )

        results.append({
            "Model": name,

            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),

            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),

            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),

            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),

            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )


In [8]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = target_col

model_order = ["TabDDPM", "ForestDiffusion"]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=cancer_data,
    test_df=cancer_data,
    label_col=target_col,
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not trained")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=cancer_data,
        label_col=target_col,
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.9789 ± 0.0098,0.9708 ± 0.0138,0.9902 ± 0.0120,0.9524 ± 0.0213
9,MLP,0.9746 ± 0.0133,0.9646 ± 0.0187,0.9855 ± 0.0192,0.9452 ± 0.0302
6,ExtraTrees,0.9702 ± 0.0119,0.9590 ± 0.0165,0.9715 ± 0.0251,0.9476 ± 0.0278
1,SVM-RBF,0.9684 ± 0.0181,0.9563 ± 0.0250,0.9757 ± 0.0281,0.9381 ± 0.0305
8,AdaBoost,0.9667 ± 0.0102,0.9538 ± 0.0144,0.9735 ± 0.0221,0.9357 ± 0.0283
7,GradientBoost,0.9632 ± 0.0203,0.9494 ± 0.0278,0.9620 ± 0.0358,0.9381 ± 0.0340
5,RandomForest,0.9623 ± 0.0136,0.9482 ± 0.0184,0.9618 ± 0.0297,0.9357 ± 0.0214
2,KNN,0.9623 ± 0.0188,0.9471 ± 0.0267,0.9751 ± 0.0246,0.9214 ± 0.0399
4,DecisionTree,0.9351 ± 0.0205,0.9122 ± 0.0277,0.9097 ± 0.0373,0.9167 ± 0.0416
3,NaiveBayes,0.9325 ± 0.0232,0.9056 ± 0.0328,0.9310 ± 0.0364,0.8833 ± 0.0516


TabDDPM - TSTR


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
1,SVM-RBF,0.6202 ± 0.0118,0.0232 ± 0.0567,0.0705 ± 0.1481,0.0143 ± 0.0356
0,LogReg,0.5956 ± 0.0476,0.0724 ± 0.0916,0.2006 ± 0.2142,0.0548 ± 0.0845
9,MLP,0.5807 ± 0.0754,0.4871 ± 0.0925,0.4809 ± 0.1092,0.5810 ± 0.2274
4,DecisionTree,0.5746 ± 0.1279,0.2156 ± 0.2093,0.4396 ± 0.2910,0.2762 ± 0.3517
5,RandomForest,0.5518 ± 0.0858,0.2617 ± 0.2134,0.2913 ± 0.1605,0.3119 ± 0.3425
6,ExtraTrees,0.5026 ± 0.0799,0.4512 ± 0.2279,0.3244 ± 0.1642,0.7429 ± 0.3757
7,GradientBoost,0.4886 ± 0.0820,0.2293 ± 0.1701,0.2283 ± 0.1349,0.2595 ± 0.2415
3,NaiveBayes,0.4772 ± 0.1403,0.5679 ± 0.0566,0.4661 ± 0.1893,0.9238 ± 0.1895
8,AdaBoost,0.4754 ± 0.1522,0.2715 ± 0.2361,0.2335 ± 0.1489,0.4310 ± 0.4202
2,KNN,0.2289 ± 0.0476,0.2501 ± 0.0881,0.1922 ± 0.0580,0.3643 ± 0.1667


TabDDPM - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
0,LogReg,0.9825 ± 0.0111,0.9754 ± 0.0159,1.0000 ± 0.0000,0.9524 ± 0.0301
8,AdaBoost,0.9772 ± 0.0137,0.9684 ± 0.0189,0.9863 ± 0.0268,0.9524 ± 0.0353
1,SVM-RBF,0.9763 ± 0.0142,0.9669 ± 0.0202,0.9877 ± 0.0123,0.9476 ± 0.0350
7,GradientBoost,0.9728 ± 0.0149,0.9620 ± 0.0212,0.9852 ± 0.0163,0.9405 ± 0.0357
9,MLP,0.9711 ± 0.0096,0.9598 ± 0.0137,0.9804 ± 0.0145,0.9405 ± 0.0244
5,RandomForest,0.9693 ± 0.0212,0.9562 ± 0.0308,0.9899 ± 0.0164,0.9262 ± 0.0527
6,ExtraTrees,0.9684 ± 0.0201,0.9549 ± 0.0293,0.9922 ± 0.0119,0.9214 ± 0.0500
2,KNN,0.9526 ± 0.0185,0.9332 ± 0.0275,0.9629 ± 0.0227,0.9071 ± 0.0505
3,NaiveBayes,0.9281 ± 0.0211,0.8999 ± 0.0308,0.9198 ± 0.0338,0.8833 ± 0.0558
4,DecisionTree,0.9184 ± 0.0236,0.8875 ± 0.0346,0.8979 ± 0.0258,0.8786 ± 0.0527


ForestDiffusion - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,ForestDiffusion,LogReg,-0.003509,-0.004578,-0.009823,0.000000e+00,0.9789 ± 0.0098,0.9825 ± 0.0111
1,ForestDiffusion,MLP,0.003509,0.004815,0.005053,4.761905e-03,0.9746 ± 0.0133,0.9711 ± 0.0096
2,ForestDiffusion,ExtraTrees,0.001754,0.004106,-0.020691,2.619048e-02,0.9702 ± 0.0119,0.9684 ± 0.0201
3,ForestDiffusion,SVM-RBF,-0.007895,-0.010637,-0.011987,-9.523810e-03,0.9684 ± 0.0181,0.9763 ± 0.0142
4,ForestDiffusion,AdaBoost,-0.010526,-0.014597,-0.012798,-1.666667e-02,0.9667 ± 0.0102,0.9772 ± 0.0137
5,ForestDiffusion,GradientBoost,-0.009649,-0.012536,-0.023201,-2.380952e-03,0.9632 ± 0.0203,0.9728 ± 0.0149
6,ForestDiffusion,RandomForest,-0.007018,-0.008012,-0.028104,9.523810e-03,0.9623 ± 0.0136,0.9693 ± 0.0212
7,ForestDiffusion,KNN,0.009649,0.013843,0.012236,1.428571e-02,0.9623 ± 0.0188,0.9526 ± 0.0185
8,ForestDiffusion,DecisionTree,0.016667,0.024788,0.011844,3.809524e-02,0.9351 ± 0.0205,0.9184 ± 0.0236
9,ForestDiffusion,NaiveBayes,0.004386,0.005675,0.011257,-2.220446e-16,0.9325 ± 0.0232,0.9281 ± 0.0211


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,-0.000263,0.000287,-0.006621,0.006429
1,TabDDPM,0.451842,0.663684,0.670880,0.535476


In [9]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
